# 01. Pemeriksaan Dataset

Notebook ini memeriksa dataset **Diabetes Health Indicators (BRFSS 2015)** sebelum preprocessing dan pemodelan.

Isi dan logika pemeriksaan mengikuti script sumber `check_data.py`.

## 0. Mengatur Folder Kerja

Notebook ini disimpan di folder `notebooks`, sedangkan data berada di folder `glysense_backend`. Sel berikut memindahkan folder kerja satu tingkat ke atas agar jalur seperti `data/...` ditemukan. Aman dijalankan berulang.

In [1]:
import os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print(os.getcwd())

d:\Tugas Akhir\Aplikasi Diabetes\glysense_backend


## 1. Memuat Dataset

Jalur file mengasumsikan notebook dijalankan dari direktori utama `glysense_backend`.

In [2]:
import pandas as pd

file_path = "data/diabetes_012_health_indicators_BRFSS2015.csv"

df = pd.read_csv(file_path)

target_col = "Diabetes_012"

## 2. Memeriksa Jumlah Data dan Fitur

Kolom target `Diabetes_012` tidak dihitung sebagai fitur prediktor.

In [3]:
feature_cols = [
    col for col in df.columns
    if col != target_col
]

print("Jumlah data:")
print(df.shape)

print("\nJumlah fitur:")
print(len(feature_cols))

print("\nNama fitur:")
print(feature_cols)

Jumlah data:
(253680, 22)

Jumlah fitur:
21

Nama fitur:
['HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker', 'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth', 'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education', 'Income']


## 3. Memeriksa Kombinasi Fitur dengan Target Berbeda

`groupby(feature_cols).nunique()` menghitung jumlah label target unik pada setiap kombinasi nilai fitur. Jika jumlahnya lebih dari satu, kombinasi fitur tersebut muncul dengan target yang berbeda.

In [4]:
print("\nMengecek apakah kombinasi 21 fitur yang sama memiliki target berbeda...")

target_variation = (
    df.groupby(feature_cols)[target_col]
    .nunique()
)

conflicting_patterns = target_variation[
    target_variation > 1
]

print("\nJumlah kombinasi fitur yang memiliki target berbeda:")
print(len(conflicting_patterns))


Mengecek apakah kombinasi 21 fitur yang sama memiliki target berbeda...

Jumlah kombinasi fitur yang memiliki target berbeda:
1834


## 4. Menampilkan Contoh Jika Ditemukan Konflik

Jika ada kombinasi fitur identik dengan target berbeda, tampilkan satu contoh baris data dan jumlah masing-masing label target.

In [5]:
if len(conflicting_patterns) > 0:
    print("\nContoh kombinasi fitur yang memiliki target berbeda:")

    example_index = conflicting_patterns.index[0]

    if not isinstance(example_index, tuple):
        example_index = (example_index,)

    condition = pd.Series(True, index=df.index)

    for col, value in zip(feature_cols, example_index):
        condition &= df[col] == value

    example_rows = df[condition]

    print(example_rows)

    print("\nTarget yang muncul pada contoh tersebut:")
    print(example_rows[target_col].value_counts().sort_index())

else:
    print("\nTidak ditemukan kombinasi 21 fitur yang sama dengan target berbeda.")


Contoh kombinasi fitur yang memiliki target berbeda:
        Diabetes_012  HighBP  HighChol  CholCheck   BMI  Smoker  Stroke  \
36786            1.0     0.0       0.0        1.0  18.0     0.0     0.0   
103540           0.0     0.0       0.0        1.0  18.0     0.0     0.0   

        HeartDiseaseorAttack  PhysActivity  Fruits  ...  AnyHealthcare  \
36786                    0.0           1.0     1.0  ...            1.0   
103540                   0.0           1.0     1.0  ...            1.0   

        NoDocbcCost  GenHlth  MentHlth  PhysHlth  DiffWalk  Sex  Age  \
36786           0.0      2.0       2.0       0.0       0.0  0.0  2.0   
103540          0.0      2.0       2.0       0.0       0.0  0.0  2.0   

        Education  Income  
36786         6.0     8.0  
103540        6.0     8.0  

[2 rows x 22 columns]

Target yang muncul pada contoh tersebut:
Diabetes_012
0.0    1
1.0    1
Name: count, dtype: int64


## 5. Catatan Pemeriksaan

Notebook ini hanya memeriksa struktur dataset dan konsistensi kombinasi fitur terhadap label target. Tidak ada data yang dihapus atau diubah, dan belum dilakukan preprocessing, pembagian dataset, maupun pelatihan model.